In [1]:
import pandas as pd
import numpy as np
import pickle, math, glob, os
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score

In [2]:
print("V9 COMPLETE - Win + Correct Score + Over/Under - Pure Poisson")

DATA_FOLDER = r"C:/Users/user/Desktop/Premier league/Data"
files = sorted(glob.glob(DATA_FOLDER + "/*.csv"))

df_list = []
for i, f in enumerate(files):
    tmp = pd.read_csv(f).copy()
    tmp['Season'] = i
    df_list.append(tmp)
df = pd.concat(df_list, ignore_index=True)
df = df[['Season','Date','HomeTeam','AwayTeam','FTHG','FTAG']].dropna()
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed')
df = df.sort_values(['Season','Date']).reset_index(drop=True)
df['TotalGoals'] = df['FTHG'] + df['FTAG']
df['Over25'] = (df['TotalGoals'] > 2.5).astype(int)
df['Result'] = np.where(df['FTHG'] > df['FTAG'], 'H', np.where(df['FTHG'] == df['FTAG'], 'D', 'A'))
df['CorrectScore_Cap'] = df['FTHG'].clip(upper=4).astype(str) + "-" + df['FTAG'].clip(upper=4).astype(str)

def poisson(lam,k): return (math.exp(-lam)*(lam**k))/math.factorial(k)

def get_probs(ph, pa):
    p0 = poisson(ph,0)*poisson(pa,0)
    p1 = poisson(ph,1)*poisson(pa,0) + poisson(ph,0)*poisson(pa,1)
    p2 = poisson(ph,2)*poisson(pa,0) + poisson(ph,1)*poisson(pa,1) + poisson(ph,0)*poisson(pa,2)
    over_prob = 1 - p0 - p1 - p2
    p_home = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h>a)
    p_draw = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h==a)
    p_away = sum(poisson(ph,h)*poisson(pa,a) for h in range(6) for a in range(6) if h<a)
    total = p_home + p_draw + p_away
    p_home, p_draw, p_away = p_home/total, p_draw/total, p_away/total
    best_score = max([(h,a) for h in range(5) for a in range(5)], key=lambda x: poisson(ph,x[0])*poisson(pa,x[1]))
    best_score_str = f"{best_score[0]}-{best_score[1]}"
    return over_prob, p_home, p_draw, p_away, best_score_str

last_100 = df.tail(100)
league_home_avg = last_100['FTHG'].mean()
league_away_avg = last_100['FTAG'].mean()
print(f"\nLAST 100 GAMES: Home {league_home_avg:.2f} Away {league_away_avg:.2f} Total {league_home_avg+league_away_avg:.2f}")
attack = {}; defence = {}


V9 COMPLETE - Win + Correct Score + Over/Under - Pure Poisson

LAST 100 GAMES: Home 1.53 Away 1.20 Total 2.73


In [3]:
# FIXED - Separate float and string columns
df['Poisson_Home'] = 0.0
df['Poisson_Away'] = 0.0
df['Poisson_Over_Prob'] = 0.0
df['Pred_H'] = 0.0
df['Pred_D'] = 0.0
df['Pred_A'] = 0.0
df['Pred_CS'] = "0-0"

for season, group in df.groupby('Season'):
    for idx in group.index:
        row = df.loc[idx]; ht, at = row['HomeTeam'], row['AwayTeam']
        if ht not in attack: attack[ht]=[league_home_avg]; defence[ht]=[league_away_avg]
        if at not in attack: attack[at]=[league_away_avg]; defence[at]=[league_home_avg]
        ha = np.mean(attack[ht][-10:]); hd = np.mean(defence[ht][-10:])
        aa = np.mean(attack[at][-10:]); ad = np.mean(defence[at][-10:])
        ph = (ha / league_home_avg) * (ad / league_away_avg) * league_home_avg
        pa = (aa / league_away_avg) * (hd / league_home_avg) * league_away_avg
        ph = 0.8*ph + 0.2*league_home_avg
        pa = 0.8*pa + 0.2*league_away_avg
        ph = max(0.3, min(3.0, ph)); pa = max(0.2, min(2.6, pa))
        over_prob, p_h, p_d, p_a, best_cs = get_probs(ph, pa)
        df.at[idx,'Poisson_Home']=ph
        df.at[idx,'Poisson_Away']=pa
        df.at[idx,'Poisson_Over_Prob']=over_prob
        df.at[idx,'Pred_H']=p_h; df.at[idx,'Pred_D']=p_d; df.at[idx,'Pred_A']=p_a
        df.at[idx,'Pred_CS']=best_cs
        attack[ht].append(row['FTHG']); defence[ht].append(row['FTAG'])
        attack[at].append(row['FTAG']); defence[at].append(row['FTHG'])



In [4]:
df_test = df.tail(100).copy()
print(f"\n===== TESTING ON 2627 (100 games) =====")

df_test['Pred_Result'] = df_test[['Pred_H','Pred_D','Pred_A']].idxmax(axis=1).map({'Pred_H':'H','Pred_D':'D','Pred_A':'A'})
win_acc = (df_test['Pred_Result'] == df_test['Result']).mean()*100
home_baseline = (df_test['Result']=='H').mean()*100
print(f"\n--- WIN (1X2) ---")
print(f"Baseline (Always Home Win): {home_baseline:.1f}%")
print(f"V9 WIN Accuracy: {win_acc:.1f}%")
for thresh in [0.55, 0.60]:
    mask = df_test['Pred_H'] > thresh
    if mask.sum() >= 5:
        acc = (df_test.loc[mask,'Result']=='H').mean()*100
        print(f"  Home Win > {thresh*100:.0f}%: {mask.sum()} games, Home Win Accuracy {acc:.1f}%")
for thresh in [0.35, 0.40]:
    mask = df_test['Pred_D'] > thresh
    if mask.sum() >= 5:
        acc = (df_test.loc[mask,'Result']=='D').mean()*100
        print(f"  Draw > {thresh*100:.0f}%: {mask.sum()} games, Draw Accuracy {acc:.1f}%")

print(f"\n--- OVER / UNDER ---")
print(f"Baseline Over: {df_test['Over25'].mean()*100:.1f}% | Baseline Under: {(1-df_test['Over25'].mean())*100:.1f}%")




===== TESTING ON 2627 (100 games) =====

--- WIN (1X2) ---
Baseline (Always Home Win): 42.0%
V9 WIN Accuracy: 46.0%
  Home Win > 55%: 28 games, Home Win Accuracy 57.1%
  Home Win > 60%: 19 games, Home Win Accuracy 63.2%

--- OVER / UNDER ---
Baseline Over: 47.0% | Baseline Under: 53.0%


In [5]:
pred_over = (df_test['Poisson_Over_Prob'] > 0.5).astype(int)
print(f"V9 OVER Accuracy (Prob >50%): {(pred_over == df_test['Over25']).mean()*100:.1f}%")
for thresh in [0.60, 0.65]:
    mask = df_test['Poisson_Over_Prob'] > thresh
    if mask.sum() >= 5:
        print(f"  Over > {thresh*100:.0f}%: {mask.sum()} games, Accuracy {(df_test.loc[mask,'Over25']==1).mean()*100:.1f}%")
for thresh in [0.40, 0.35]:
    mask = df_test['Poisson_Over_Prob'] < thresh
    if mask.sum() >= 5:
        print(f"  Under (OverProb < {thresh*100:.0f}%): {mask.sum()} games, Under Accuracy {(df_test.loc[mask,'Over25']==0).mean()*100:.1f}%")

print(f"\n--- CORRECT SCORE ---")
cs_acc = (df_test['Pred_CS'] == df_test['CorrectScore_Cap']).mean()*100
print(f"V9 CORRECT SCORE: {cs_acc:.1f}%")
print(f"Baseline (Always 1-0): {(df_test['CorrectScore_Cap']=='1-0').mean()*100:.1f}%")


V9 OVER Accuracy (Prob >50%): 51.0%
  Over > 60%: 39 games, Accuracy 48.7%
  Over > 65%: 24 games, Accuracy 41.7%
  Under (OverProb < 40%): 14 games, Under Accuracy 64.3%

--- CORRECT SCORE ---
V9 CORRECT SCORE: 11.0%
Baseline (Always 1-0): 7.0%


In [6]:
pickle.dump(attack, open('attack.pkl','wb'))
pickle.dump(defence, open('defence.pkl','wb'))
pickle.dump({'home_avg': league_home_avg, 'away_avg': league_away_avg}, open('league_avg.pkl','wb'))
df.to_csv('processed_history.csv', index=False)
print("\nDONE V9 COMPLETE - Win + Over/Under + Correct Score saved.")


DONE V9 COMPLETE - Win + Over/Under + Correct Score saved.
